In [1]:
%load_ext autoreload
%autoreload 2



In [ ]:

import os
import sys

import datetime
import requests
import re
import pathlib
import numpy as np
import time
import dijkstra3d
import pyproj
import ipyleaflet

from shapely.geometry import shape
import geopandas as gpd
import pandas as pd

import rasterio
from scipy.signal import find_peaks
from scipy import interpolate
import matplotlib.pyplot as plt
from scipy import ndimage

import livemaptiles
from matplotlib import cm

import skfmm
from skimage.transform import resize
from types import SimpleNamespace

import fiona
#force geopandas to support kml
fiona.supported_drivers['KML'] = 'rw'
# gpd.io.file.fiona.drvsupport.suppor# ted_drivers['KML'] = 'rw'

import site
# site.addsitedir(r'C:\Users\rea095\Documen
# site.addsitedir(r'\\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work\3_New_Bradfield_2020\1_Project_management\8_Special_issue\2_WaterRoute\3_Model\waterroute\waterroute\tiled_flow')
site.addsitedir('waterroute\tiled_flow')
import drain

from waterroute import utils
from waterroute import channelroute
from waterroute.channel import channelcost



In [ ]:

dempath = "inputs/DEM/merged_dem.tif"
rockDepthPath = "inputs/DEM/low_res_resampled.tif"
sunwaterPath = "inputs/DEM/rasterize_sunwater_route.tif"

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*Lossy conversion occurred.*")


def removeLayer(theMap,name):
    for l in theMap.layers:
        if l.name == name:
            theMap.remove_layer(l)

            
def addLayer(theMap,name,array,aff,crs,scale_min,scale_max,colourMap = cm.autumn):
    removeLayer(theMap,name)
    at = livemaptiles.arrayTile(array,aff,crs)
    at.scale_min = scale_min
    at.scale_max = scale_max

    at.colourMap = colourMap
    at.colourMap.set_under(alpha=0)
    at.colourMap.set_over(alpha=0)
    livemaptiles.tileServer.layers[name] = at.tileMaker

    tile_layer=ipyleaflet.TileLayer(name=name, url=f'http://localhost:8080/livemaptiles/{name}/{{z}}/{{x}}/{{y}}.png')
    theMap.add_layer(tile_layer)



def start_map(start = [145.829917,  -24.693563]):
    m = ipyleaflet.Map(center=start[::-1] , zoom=8,)
    m.scroll_wheel_zoom = True
    search = ipyleaflet.SearchControl(position='topleft', 
                 url= 'https://nominatim.openstreetmap.org/search?format=json&q={s}', 
                 #zoom=5,
                 property_name='display_name',
                 marker=ipyleaflet.Marker()
                 )
    m.add_control(search)

    m.add_control(ipyleaflet.FullScreenControl())

    m.add_control(ipyleaflet.LayersControl())

    measure = ipyleaflet.MeasureControl(
        position='topleft',
        active_color = 'orange',
        primary_length_unit = 'kilometers',
        secondary_length_unit = 'meters',
        primary_area_unit = 'sqmeters'
    )
    m.add_control(measure)

    t = ipyleaflet.basemap_to_tiles(ipyleaflet.basemaps.OpenTopoMap)
    t.base = True
    m.add_layer(t)
    return m

m = start_map()
m

def outLocation(run_name):
    #base_dir = r'\\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work\2_Old_Bradfield_2020\99_Working\5_rea095\channel'
    #base_dir = r'\\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work\3_New_Bradfield_2020\99_Working\9_rea095\channel'
    # base_dir = r'\\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work\3_New_Bradfield_2020\3_Infrastructure\2_Diversions\2_waterroute\routes'
    base_dir = r'\\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work\3_New_Bradfield_2020\1_Project_management\8_Special_issue\2_WaterRoute\3_Model\1_Routes'
    return utils.out_location(base_dir,run_name)



def runner(CR, dem_ratio = 1/1,dem=None,aff=None,crs=None):
    if dem is None or aff is None or crs is None:
        dem,aff,crs = utils.load_raster(dempath,CR.routing_bounds,1,dem_ratio,utils.Resampling.average)
    addLayer(m,f'dem',dem,aff,crs,200,1000,cm.viridis)
    rockDepth, _, _ = utils.load_raster(rockDepthPath,CR.routing_bounds,1,dem_ratio,utils.Resampling.average)
    

    #save rockDepth as pickel for debugging
    # with open(f'{outLocation(CR.routing_out_name)}_rockDepth.pkl', 'wb') as f:
    #     pickle.dump(rockDepth, f)
    # #save dem as pickel for debugging
    # with open(f'{outLocation(CR.routing_out_name)}_dem.pkl', 'wb') as f:
    #     pickle.dump(dem, f)
    
    output_path = outLocation(CR.routing_out_name)

    paths = channelroute.elevRangePaths(output_path,
                                        dem,
                                        aff,
                                        crs,
                                        CR.routing_start_elevation,
                                        CR.routing_start_elevation+1,
                                        # CR.routing_end_elevation,

                                        targetSlope=CR.routing_targetSlope,
                                        start=CR.routing_start,
                                        finish=CR.routing_finish,
                                        max_fill_siphon_ends = 5,
                                        contour=CR.routing_contour,
                                        doCrossDrainage=CR.routing_doCrossDrainage,
                                        sampleF = 250,
                                        bed_width = CR.routing_bed_width,
                                        water_depth = CR.routing_water_depth,
                                        freeboard = CR.routing_freeboard,
                                        additional_per_m_cost = CR.routing_additional_per_m_cost, 
                                        channel_capacity = CR.gross_diversions_annual_m3_per_s, 
                                        rockDepth_raster = rockDepth, 
                                        loss_value_dollar_per_ML = CR.loss_value_dollar_per_ML,
                                        total_linear_cost = CR.total_linear_cost,
                                        total_loss_GL = CR.total_loss_GL,
                                        no_cells = CR.no_cells, 
                                        withLengthPenalty = CR.withLengthPenalty,
                                        cost_array_with_rock_depth = CR.cost_array_with_rock_depth,
                                        sunWater_route = CR.sunWater_route,
                                        approximate_length = CR.approximate_length
                                        )

    path,profile,l = paths[list(paths.keys())[0]]
    CR.profile = profile.copy()
    # #save CR as pickle for debugging
    # import pickle
    # with open(f'{output_path}_CR.pkl', 'wb') as f:        pickle.dump(CR, f)
    #cost_calc aims to get the detailed cost by adjusting the distance of each reach
    channelcost.costs_calc(CR)
    CR.profile.to_csv(f'{output_path}_full_profile.csv')
    d = CR.__dict__.copy()
    del d['reaches']
    del d['profile']
    pd.DataFrame([d]).T.to_csv(f'{output_path}_costs.csv')
    
    pathDF = gpd.GeoDataFrame([l], columns=['geometry'])
    path_layer = ipyleaflet.GeoData(geo_dataframe=pathDF, style={'color':'blue', 'weight':4})
    path_layer.name = str(output_path)
    removeLayer(m,path_layer.name)
    m.add_layer(path_layer)
    
    # global last_output_path
    # last_output_path = output_path
    return(output_path)




# Patch livemaptiles to add pyproj Transformer to fix argument error, 08012026
from pyproj import Transformer
import livemaptiles

def new_arrayTile_init(self, in_array, in_affine, in_crs):
    self.in_array = in_array
    self.in_affine = in_affine
    self.in_crs = in_crs
    self.tile_size = (256, 256)
    
    # We do not need to assign self.tileMaker if it is already a method on the class.
    # self.tileMaker exists as a method.
    
    self.colourMap = cm.autumn
    self.scale_min = self.in_array.min()
    self.scale_max = self.in_array.max()
    self.alpha = None
    # EPSG:4326 wgs84, 3857 is spherical mercator
    # Removed skip_equivalent=True which was causing the error
    self.transformer = Transformer.from_crs(3857, self.in_crs, always_xy=True)

livemaptiles.arrayTile.__init__ = new_arrayTile_init


from waterroute.channel import Losses
from scipy.optimize import root_scalar
# ---- solve channel dimension ----
def solve_depth_ratio(Q, S, n, ratio):
    S = abs(S) 
    def residual(d):

        width = ratio * d
        
        Q_calc, V, A = trapezoidal_discharge(width=width, depth=d, S=S, n=n)

        return Q_calc - Q

    sol = root_scalar(residual, bracket=[0.1, 10], method='brentq')

    return sol.root


from scipy.optimize import root_scalar
import numpy as np


def trapezoidal_discharge(width, depth, S, n = 0.03, z=2):
    """
    Calculate discharge for a trapezoidal channel using Manning's equation.

    Parameters
    ----------
    width : float
        Bottom width (m)
    depth : float
        Flow depth (m)
    S : float
        Channel slope (m/m)
    n : float
        Manning roughness coefficient
    z : float, optional
        Side slope (horizontal : vertical). Default = 2

    Returns
    -------
    Q : float
        Discharge (m3/s)
    V : float
        Velocity (m/s)
    """

    # Cross-sectional area
    A = depth * (width + z * depth)

    # Wetted perimeter
    P = width + 2 * depth * np.sqrt(1 + z**2)

    # Hydraulic radius
    R = A / P

    # Velocity (Manning)
    V = (1 / n) * (R ** (2/3)) * (S ** 0.5)

    # Discharge
    Q = A * V

    return Q, V, A



def run_channel(section = 'section1', ratio_width_depth=5, run_name=None, target_slope = None, bedwidth = None, waterdepth = None, linear_cost = None,
                 loss_value_dollar_per_ML = 75, is_length_penalty =True, cost_array_with_rock_depth= True,sunWater_route = False):

    CR = channelcost.Channel_Route()

    left = 147
    top = -19
    right = 149
    bottom = -21
    start = (147.31,  -19.81) #start
    test_end = (147.50,  -19.900) #test end
    end1 = (147.857,  -19.978) #end1, before pumping station
    end1_2 = (147.866,  -19.974) #start of section2, after pumping station
    end2_5 = (148.079,  -20.086)#end of section2, befure Euri creek

    CR.routing_bounds = (left, bottom, right, top)
    CR.routing_out_name = run_name
    # CR.routing_targetSlope = -abs(initial_slope)
    if (target_slope is not None):
        CR.routing_targetSlope = -abs(target_slope)
        
    if (section=='section1'):
        CR.routing_start = start
        CR.routing_finish = end1
        #This is DFL
        CR.routing_start_elevation = 36
        CR.gross_diversions_annual_m3_per_s = 19.4
        if linear_cost is not None:
            CR.total_linear_cost = linear_cost
        else:
            # CR.total_linear_cost = 13424602
            CR.total_linear_cost = 583930
        # CR.total_linear_cost = 0
        CR.total_loss_GL = 8.7  
        CR.approximate_length = 67 #km, this is the approximate length of the channel, used to calculate length penalty
        CR.no_cells = 1991
        CR.loss_value_dollar_per_ML = loss_value_dollar_per_ML
        CR.cost_array_with_rock_depth = cost_array_with_rock_depth
        CR.withLengthPenalty = is_length_penalty

    elif (section=='section2'):
        CR.routing_start = end1_2
        CR.routing_finish = end2_5
        #This is DFL
        CR.routing_start_elevation = 40    
        CR.gross_diversions_annual_m3_per_s = 10
        if linear_cost is not None:
            CR.total_linear_cost = linear_cost
        else:
            CR.total_linear_cost = 577045
        CR.total_loss_GL = 2.9
        CR.approximate_length = 40
        CR.no_cells = 805
        CR.loss_value_dollar_per_ML = loss_value_dollar_per_ML
        CR.cost_array_with_rock_depth = cost_array_with_rock_depth
        CR.withLengthPenalty = is_length_penalty
        
    CR.sunWater_route = sunWater_route
    CR.ratio_width_depth = ratio_width_depth
    #set initial slope to be positive
    # CR.routing_water_depth = solve_depth_ratio(Q=CR.gross_diversions_annual_m3_per_s, S= abs(initial_slope), n=0.03, ratio=CR.ratio_width_depth)
    # CR.routing_bed_width = CR.ratio_width_depth * CR.routing_water_depth
    CR.routing_water_depth = waterdepth
    CR.routing_bed_width = bedwidth

    print(f"Calculated channel dimensions: slope = {CR.routing_targetSlope}, ratio_width_depth = {CR.ratio_width_depth}, water depth = {CR.routing_water_depth:.2f} m, bed width = {CR.routing_bed_width:.2f} m")

    CR.routing_contour = 27
    # ---- other parameters ----
    CR.routing_doCrossDrainage = True
    CR.routing_freeboard = 1
    CR.routing_additional_per_m_cost = 2000
    CR.lined = False

    CR.life_span_years = 50
    CR.discount_rate = 0.07
    CR.rockDepth = 5 #not use this anymore
    #maybe delete this parameter, I already insert the function that calculate actual water delivered based on channel capacity of m^3/s
    CR.gross_diversions_annual_l = 608.645 * 1E9 #this is based on 19.4 m3/s, which is the current gross diversion, but we may want to adjust this for future scenarios

    CR.railway_crossing_count = 0
    CR.highway_crossings_count = 0
    CR.distance_between_farm_crossing_km = 4
    CR.rolling_shutdown_factor = 2/3
    CR.large_terminal_storage = True
    CR.max_reach_length_m = 200000

    CR.maxControlSpacing = 28000
    CR.real_channelSlope = 1/12000
    CR.bay_minWaterDepth = 1

    output_path = runner(CR=CR, dem_ratio=1)
    return(output_path)

In [3]:
import pandas as pd

# width_to_depth_ratio = 5
# run_name = 'test6-stiffer-slope'
# section = 'section1'
# slope_for_channel_dimension = 1/15000    

def sequential_run(number_of_runs = 3, width_to_depth_ratio = None, water_depth = None, bed_width = None, run_name = None, section = 'section1', 
                   slope_for_channel_dimension = 1/15000, linear_cost = None, loss_value_dollar_per_ML = 75, is_length_penalty=True, cost_array_with_rock_depth = True):
    
    if section == 'section1':
        channel_capacity_m3_sec = 19.4
    elif section == 'section2':
        channel_capacity_m3_sec = 10
    
    def get_cost_scalar(df, key):
        v = df.loc[key]
        if isinstance(v, pd.Series):
            v = v.dropna().iloc[0] if not v.dropna().empty else np.nan
        return pd.to_numeric(v, errors="coerce")

    template_sequential_run_df = pd.read_csv(r"\\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work\3_New_Bradfield_2020\1_Project_management\8_Special_issue\2_WaterRoute\3_Model\Sequential_Run.csv")
    sequential_run_df = template_sequential_run_df.copy()
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'slope'] = slope_for_channel_dimension 
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'width_to_depth_ratio'] = width_to_depth_ratio
    # water_depth = solve_depth_ratio(Q=channel_capacity_m3_sec, S= slope_for_channel_dimension, n=0.03, ratio=width_to_depth_ratio)
    # bed_width = width_to_depth_ratio * water_depth
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'water_depth'] = water_depth
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'bed_width'] = bed_width
    ############   #model run 1   ##################################
    output_path_1 = run_channel(section = section, ratio_width_depth=width_to_depth_ratio, run_name= run_name, target_slope = slope_for_channel_dimension, bedwidth = bed_width,
                                 waterdepth = water_depth, linear_cost = linear_cost,loss_value_dollar_per_ML = loss_value_dollar_per_ML, is_length_penalty=is_length_penalty,
                                  cost_array_with_rock_depth=cost_array_with_rock_depth, sunWater_route = False)

    #collecti run 1 results
    cost_df = pd.read_csv(f'{output_path_1}_costs.csv', index_col=0)
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'water_surface_width'] = abs(get_cost_scalar(cost_df, 'water_surface_width'))
    _, V, _ = channelroute.trapezoidal_discharge(width=abs(get_cost_scalar(cost_df, 'water_surface_width')), depth=water_depth, S=slope_for_channel_dimension, n=0.03, z=2)
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'velocity'] = V
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'final_slope'] = abs(get_cost_scalar(cost_df, 'final_slope'))
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', '1inX_final_slope'] = 1/abs(get_cost_scalar(cost_df, 'final_slope'))
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'total_length'] = get_cost_scalar(cost_df, 'total_length_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'elevation_difference'] = get_cost_scalar(cost_df, 'total_elevation_change_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'head_loss_to_consider'] = get_cost_scalar(cost_df, 'total_controlled_head_loss_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'adjusted_slope'] = abs(get_cost_scalar(cost_df, 'adjusted_final_slope'))
    sequential_run_df.loc[sequential_run_df['run_ID']=='1', '1inX_adjusted_slope'] = 1/abs(get_cost_scalar(cost_df, 'adjusted_final_slope'))
    sequential_run_df['cost_file_name'] = sequential_run_df['cost_file_name'].astype('string')
    sequential_run_df.loc[sequential_run_df['run_ID'] == '1', 'cost_file_name'] = f'{output_path_1}_costs.csv'
    sequential_run_df.to_csv(f'{output_path_1}_seq1.csv', index=False)
    # if number_of_runs is 1 stop here
    if number_of_runs == 1:
        return 
    

    #run 2 with adjusted slope from run 1
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'width_to_depth_ratio'] = width_to_depth_ratio
    slope2 = sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'adjusted_slope'].values[0] #S3 -1
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'slope'] = slope2
    water_depth_2 = solve_depth_ratio(Q=channel_capacity_m3_sec, S= slope2, n=0.03, ratio=width_to_depth_ratio)
    bed_width_2 = width_to_depth_ratio * water_depth_2
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'water_depth'] = water_depth_2
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'bed_width'] = bed_width_2
    sequential_run_df
    ############   #model run  2 ##################################
    output_path_2 = run_channel(section = section, ratio_width_depth=width_to_depth_ratio, run_name= run_name,  target_slope = slope_for_channel_dimension,
                                 bedwidth = bed_width_2, waterdepth = water_depth_2, linear_cost = linear_cost,loss_value_dollar_per_ML = loss_value_dollar_per_ML,
                                   is_length_penalty=is_length_penalty, cost_array_with_rock_depth=cost_array_with_rock_depth, sunWater_route = False)

    #collect run 2 results
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'width_to_depth_ratio'] = width_to_depth_ratio
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'slope'] = sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'adjusted_slope'].values[0]
    cost_df2 = pd.read_csv(f'{output_path_2}_costs.csv', index_col=0)
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'water_surface_width'] = abs(get_cost_scalar(cost_df2, 'water_surface_width'))
    _, V, _ = channelroute.trapezoidal_discharge(width=abs(get_cost_scalar(cost_df2, 'water_surface_width')), depth=water_depth_2, S=slope2, n=0.03, z=2)
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'velocity'] = V

    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'final_slope'] = abs(get_cost_scalar(cost_df2, 'final_slope'))
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', '1inX_final_slope'] = 1/abs(get_cost_scalar(cost_df2, 'final_slope'))
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'total_length'] = get_cost_scalar(cost_df2, 'total_length_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'elevation_difference'] = get_cost_scalar(cost_df2, 'total_elevation_change_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'head_loss_to_consider'] = get_cost_scalar(cost_df2, 'total_controlled_head_loss_m')
    #positive residual head loss should get the sloe steeper, negative residual head loss should get the slope flatter
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'residual_head_loss_to_consider'] = abs(sequential_run_df.loc[sequential_run_df['run_ID']=='1', 'head_loss_to_consider'].values[0]) - abs(sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'head_loss_to_consider'].values[0])
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'cost_file_name']  = f'{output_path_2}_costs.csv'
    #stiffer slope to consider the residual head loss
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'adjusted_slope'] = (sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'elevation_difference'] - sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'head_loss_to_consider'] + sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'residual_head_loss_to_consider'])/sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'total_length']
    sequential_run_df.loc[sequential_run_df['run_ID']=='2', '1inX_adjusted_slope'] = 1/sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'adjusted_slope']
    sequential_run_df.to_csv(f'{output_path_2}_seq2.csv', index=False)
    # if number_of_runs is 1 stop here
    # if abs(sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'residual_head_loss_to_consider'].values[0]) <= abs(sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'head_loss_to_consider'].values[0])*0.01:
    #     print("No need for run 3 as the head loss is already fully considered in run 2.")
    #     return
    
    if sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'residual_head_loss_to_consider'].item() <0.1:
        print("No need for run 3 as the residual head loss is less than 0.1 m.")
        return

    if number_of_runs == 2:
        return 

    #####3rd run with adjusted slope from run 2
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'width_to_depth_ratio'] = width_to_depth_ratio
    #S1, for channel dimension
    slope3 = sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'adjusted_slope'].values[0]
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'slope'] = slope3
    water_depth_3 = solve_depth_ratio(Q=channel_capacity_m3_sec, S= slope3, n=0.03, ratio=width_to_depth_ratio)
    bed_width_3 = width_to_depth_ratio * water_depth_3
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'water_depth'] = water_depth_3
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'bed_width'] = bed_width_3

    ############   #model run-3   ##################################
    output_path_3 = run_channel(section = section, ratio_width_depth=width_to_depth_ratio, run_name= run_name, target_slope = slope_for_channel_dimension,  bedwidth = bed_width_3,
                                 waterdepth = water_depth_3, linear_cost = linear_cost, loss_value_dollar_per_ML = loss_value_dollar_per_ML, is_length_penalty=is_length_penalty,
                                   cost_array_with_rock_depth=cost_array_with_rock_depth, sunWater_route= False)

    #collect run 3 results
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'width_to_depth_ratio'] = width_to_depth_ratio
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'slope'] = sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'adjusted_slope'].values[0]
    cost_df3 = pd.read_csv(f'{output_path_3}_costs.csv', index_col=0)
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'water_surface_width'] = abs(get_cost_scalar(cost_df3, 'water_surface_width'))
    _, V, _ = channelroute.trapezoidal_discharge(width=abs(get_cost_scalar(cost_df3, 'water_surface_width')), depth=water_depth_3, S=slope3, n=0.03, z=2)
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'velocity'] = V
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'final_slope'] = abs(get_cost_scalar(cost_df3, 'final_slope'))
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', '1inX_final_slope'] = 1/abs(get_cost_scalar(cost_df3, 'final_slope'))
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'total_length'] = get_cost_scalar(cost_df3, 'total_length_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'elevation_difference'] = get_cost_scalar(cost_df3, 'total_elevation_change_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'head_loss_to_consider'] = get_cost_scalar(cost_df3, 'total_controlled_head_loss_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'residual_head_loss_to_consider'] = abs(sequential_run_df.loc[sequential_run_df['run_ID']=='2', 'head_loss_to_consider'].values[0]) - abs(sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'head_loss_to_consider'].values[0])
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'cost_file_name']  = f'{output_path_3}_costs.csv'
    #steeper slope to consider the residual head loss
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'adjusted_slope'] = (sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'elevation_difference'] - sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'head_loss_to_consider'] + sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'residual_head_loss_to_consider'])/sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'total_length']
    sequential_run_df.loc[sequential_run_df['run_ID']=='3', '1inX_adjusted_slope'] = 1/sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'adjusted_slope']
    sequential_run_df.to_csv(f'{output_path_3}_seq3.csv', index=False)

    if sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'residual_head_loss_to_consider'].item() <0.1:
        print("No need for run 3 as the residual head loss is less than 0.1 m.")
        return

    
    #####4th run with adjusted slope from run 3rd
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'width_to_depth_ratio'] = width_to_depth_ratio
    #S1, for channel dimension
    slope4 = sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'adjusted_slope'].values[0]
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'slope'] = slope4
    water_depth_4 = solve_depth_ratio(Q=channel_capacity_m3_sec, S= slope4, n=0.03, ratio=width_to_depth_ratio)
    bed_width_4 = width_to_depth_ratio * water_depth_4
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'water_depth'] = water_depth_4
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'bed_width'] = bed_width_4

    ############   #model run-4   ##################################
    output_path_4    = run_channel(section = section, ratio_width_depth=width_to_depth_ratio, run_name= run_name, target_slope = slope_for_channel_dimension,  bedwidth = bed_width_4,
                                 waterdepth = water_depth_4, linear_cost = linear_cost, loss_value_dollar_per_ML = loss_value_dollar_per_ML, is_length_penalty=is_length_penalty,
                                   cost_array_with_rock_depth=cost_array_with_rock_depth, sunWater_route= False)

    #collect run 4 results
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'width_to_depth_ratio'] = width_to_depth_ratio
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'slope'] = sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'adjusted_slope'].values[0]
    cost_df4 = pd.read_csv(f'{output_path_4}_costs.csv', index_col=0)
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'water_surface_width'] = abs(get_cost_scalar(cost_df4, 'water_surface_width'))
    _, V, _ = channelroute.trapezoidal_discharge(width=abs(get_cost_scalar(cost_df4, 'water_surface_width')), depth=water_depth_4, S=slope4, n=0.03, z=2)
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'velocity'] = V
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'final_slope'] = abs(get_cost_scalar(cost_df4, 'final_slope'))
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', '1inX_final_slope'] = 1/abs(get_cost_scalar(cost_df4, 'final_slope'))
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'total_length'] = get_cost_scalar(cost_df4, 'total_length_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'elevation_difference'] = get_cost_scalar(cost_df4, 'total_elevation_change_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'head_loss_to_consider'] = get_cost_scalar(cost_df4, 'total_controlled_head_loss_m')
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'residual_head_loss_to_consider'] = abs(sequential_run_df.loc[sequential_run_df['run_ID']=='3', 'head_loss_to_consider'].values[0]) - abs(sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'head_loss_to_consider'].values[0])
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'cost_file_name']  = f'{output_path_4}_costs.csv'
    #steeper slope to consider the residual head loss
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'adjusted_slope'] = (sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'elevation_difference'] - sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'head_loss_to_consider'] + sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'residual_head_loss_to_consider'])/sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'total_length']
    sequential_run_df.loc[sequential_run_df['run_ID']=='4', '1inX_adjusted_slope'] = 1/sequential_run_df.loc[sequential_run_df['run_ID']=='4', 'adjusted_slope']
    sequential_run_df.to_csv(f'{output_path_4}_seq4.csv', index=False)



import numpy as np

def solve_S(Q, V, z=2, n=0.03, ratio = 5):
    d = np.sqrt(Q / (V * (ratio + z)))

    _, _, A = trapezoidal_discharge(width=ratio*d, depth=d, S=1, n=n)
    R = A / (ratio*d + 2*d*np.sqrt(1+z**2))
    S = (V * n / (R ** (2/3)))**2
    
    return round(S,7), round(d,2), round(ratio*d,2)

In [8]:
version = 'v22'
section = 'section1'

if section == 'section1':
    velocity_range = np.arange(0.490, 0.491, 0.01)      #section1
    # velocity_range = np.arange(0.485, 0.490, 0.001)  #section1
elif section == 'section2':
    # velocity_range = np.arange(0.435, 0.436, 0.001)  #section2
    velocity_range = np.arange(0.425, 0.444, 0.001)  #section2
# velocity_range = np.arange(0.499, 0.500, 0.001)  #section1
# velocity_range = np.arange(0.435, 0.436, 0.001)  #section2
import numpy as np

#write
S_depth_width = []
for ratio in [5]:
    for V in velocity_range:

        ###!!!!update Q for section 1 and section 2
        if section == 'section1':
            Q = 19.4
        elif section == 'section2':
            Q = 10
        slope, depth, width = solve_S(Q=Q, V=V, ratio = ratio)
        # print(f"S: {1/S}, depth: {d}, width: {width}")    
        #add V to S_depth_width
        S_depth_width.append((section, ratio, depth, width, int(round(1/slope,0)), round(V,3) ))
        # S_depth_width.append((1/S, d, width))
    #if diff of S, S_depth_width smaller than 500, remove the row
    i = 0
    while i < len(S_depth_width) - 1:
        if abs(S_depth_width[i][4] - S_depth_width[i+1][4]) < 100:
            S_depth_width.pop(i+1)
        else:
            i += 1
    
    
# print([(slope) for _, _, _, _, slope, _ in S_depth_width])
all_slopes = [item[5] for item in S_depth_width]
all_slopes

[np.float64(0.49)]

In [9]:
for section, ratio, depth, width, slope, V in S_depth_width:
    # print(f"S: {slope}, depth: {depth}, width: {width}")
    runName = f'{version}-{section}-r{ratio}-d{depth}-w{width}-S{slope}-v{V  :.3f}'
    # print(f"Run Name: {runName}")
    rootDir  = "\\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work\3_New_Bradfield_2020\1_Project_management\8_Special_issue\2_WaterRoute\3_Model\1_Routes"
    # workDirectory = rootDir + f'\\{runName}'
    workDirectory = os.path.join(rootDir, runName)
    if os.path.isdir(workDirectory):
        print("Exists:", workDirectory)
    else:
        print(workDirectory)
        sequential_run(number_of_runs= 3,width_to_depth_ratio= ratio, water_depth=depth, bed_width=width, run_name=runName, section = section,
                        slope_for_channel_dimension= 1/slope)


\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work_New_Bradfield_2020_Project_management\8_Special_issue_WaterRoute_Model_Routes\v22-section1-r5-d2.38-w11.89-S9814-v0.490
Calculated channel dimensions: slope = -0.00010189525168127165, ratio_width_depth = 5, water depth = 2.38 m, bed width = 11.89 m


\\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work\3_New_Bradfield_2020\1_Project_management\8_Special_issue\2_WaterRoute\3_Model\waterroute\waterroute\channelroute.py:518: RuntimeWarning: invalid value encountered in cast
  rock_i = np.rint(rockDepth_raster).astype(np.int16)
c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'v22-section1-r5-d2.38-w11.89-S9814-v0.490_36' adjusted to 'v22_section1_r5_d2.38_w11.89_S9814_v0.490_36' for XML validity.
  ogr_write(


length of 36 is 81.23km. Cost is (2020 AUD) M$19.74


c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'v22-section1-r5-d2.38-w11.89-S9814-v0.490_36_cross_drainage' adjusted to 'v22_section1_r5_d2.38_w11.89_S9814_v0.490_36_cross_drainage' for XML validity.
  ogr_write(
c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'v22-section1-r5-d2.38-w11.89-S9814-v0.490_36_cross_drainage_catchment' adjusted to 'v22_section1_r5_d2.38_w11.89_S9814_v0.490_36_cross_drainage_catchment' for XML validity.
  ogr_write(


0
Calculated channel dimensions: slope = -0.00010189525168127165, ratio_width_depth = 5, water depth = 2.65 m, bed width = 13.27 m


...

0
Calculated channel dimensions: slope = -0.00010189525168127165, ratio_width_depth = 5, water depth = 2.54 m, bed width = 12.70 m


\\fs1-cbr.nexus.csiro.au\{lw-bradfield}\work\3_New_Bradfield_2020\1_Project_management\8_Special_issue\2_WaterRoute\3_Model\waterroute\waterroute\channelroute.py:518: RuntimeWarning: invalid value encountered in cast
  rock_i = np.rint(rockDepth_raster).astype(np.int16)
c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'v22-section1-r5-d2.38-w11.89-S9814-v0.490_36' adjusted to 'v22_section1_r5_d2.38_w11.89_S9814_v0.490_36' for XML validity.
  ogr_write(


length of 36 is 81.10km. Cost is (2020 AUD) M$20.09


c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'v22-section1-r5-d2.38-w11.89-S9814-v0.490_36_cross_drainage' adjusted to 'v22_section1_r5_d2.38_w11.89_S9814_v0.490_36_cross_drainage' for XML validity.
  ogr_write(
c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'v22-section1-r5-d2.38-w11.89-S9814-v0.490_36_cross_drainage_catchment' adjusted to 'v22_section1_r5_d2.38_w11.89_S9814_v0.490_36_cross_drainage_catchment' for XML validity.
  ogr_write(


0
No need for run 3 as the residual head loss is less than 0.1 m.


In [15]:
os.path.isdir(workDirectory)

False

In [ ]:
# sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v11-reach2-ratio5-velocity0.2', section = 'section2', slope_for_channel_dimension= 1.45E-05)
# sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v11-reach2-ratio5-velocity0.25', section = 'section2', slope_for_channel_dimension= 2.63E-05)
sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v11-reach2-ratio5-velocity0.3', section = 'section2', slope_for_channel_dimension= 4.28E-05)
sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v11-reach2-ratio5-velocity0.35', section = 'section2', slope_for_channel_dimension= 6.46E-05)
sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v11-reach2-ratio5-velocity0.4', section = 'section2', slope_for_channel_dimension= 9.22E-05)
sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v11-reach2-ratio5-velocity0.45', section = 'section2', slope_for_channel_dimension= 1.263E-04)
sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v11-reach2-ratio5-velocity0.5', section = 'section2', slope_for_channel_dimension= 1.672E-04)
sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v11-reach2-ratio5-velocity0.55', section = 'section2', slope_for_channel_dimension= 2.1E-04)


In [ ]:

# sequential_run(number_of_runs= 3,width_to_depth_ratio=4, run_name='v1-reach1-ratio4', section = 'section1')
# sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v1a-reach1-ratio5', section = 'section1')
# sequential_run(number_of_runs= 3,width_to_depth_ratio=8, run_name='v1-reach1-ratio8', section = 'section1')

# sequential_run(number_of_runs= 3,width_to_depth_ratio=4, run_name='v1-reach2-ratio4', section = 'section2')
sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v1a-reach2-ratio5', section = 'section2')
# sequential_run(number_of_runs= 3,width_to_depth_ratio=8, run_name='v1-reach2-ratio8', section = 'section2')

# sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v2a-reach1-ratio5', section = 'section1', is_length_penalty=False)
sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v2a-reach2-ratio5', section = 'section2', is_length_penalty=False)
# sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v3-reach1-ratio5', section = 'section1', loss_value_dollar_per_ML= 750)
# sequential_run(number_of_runs= 3,width_to_depth_ratio=5, run_name='v3-reach2-ratio5', section = 'section2', loss_value_dollar_per_ML= 750)




In [2]:
import matplotlib.pyplot as plt
import numpy as np
def annualised_cost(cost, discount_rate = 0.07, lifespan= 50, QandM_percent=0.02):
        '''
        cost = capital cost
        discount_rate = discount rate (e.g. 0.05 for 5%)
        lifespan = lifespan of the asset in years
        QandM = ratio of capital cost for annual operation and maintenance
        '''
        return cost * (discount_rate * (1 + discount_rate) ** lifespan) / ((1 + discount_rate) ** lifespan - 1) + cost * QandM_percent

annualised_cost(577045)

53353.49387758291

In [11]:
annualised_cost(5000)


462.2992476980383